In [ ]:
from datetime import datetime
import json
import sqlite3
from flask import Flask, jsonify, request

app = Flask(__name__)

# دالة لتهيئة قاعدة البيانات وإنشاء جدول السجلات (webhook_logs)
def init_db():
    conn = sqlite3.connect('store.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS webhook_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            received_data TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    ''')
    conn.commit()
    conn.close()

# استقبال الـ POST request على الـ Endpoint المسماة /webhook
@app.route('/webhook', methods=['POST'])
def webhook():
    # التحقق من أن الطلب يحتوي على بيانات JSON
    if request.is_json:
        incoming_data = request.get_json()

        # طباعة البيانات في الـ Terminal
        print("--- تم استقبال Webhook جديد بنجاح ---")
        print(json.dumps(incoming_data, indent=4))

        # تخزين البيانات في قاعدة البيانات store.db في جدول webhook_logs
        conn = sqlite3.connect('store.db')
        cursor = conn.cursor()

        data_string = json.dumps(incoming_data)
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        cursor.execute('''
            INSERT INTO webhook_logs (received_data, timestamp)
            VALUES (?, ?)
        ''', (data_string, current_time))

        conn.commit()
        conn.close()

        # إرجاع رسالة تأكيد بسيطة
        return jsonify({"status": "received"}), 200
    else:
        return jsonify({"error": "Request must be JSON"}), 400

if __name__ == '__main__':
    init_db()  # تشغيل إنشاء الجدول قبل بدء السيرفر
    # تشغيل السيرفر المحلي على البورت الافتراضي 5000
    app.run(debug=True, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
